# Multi-scale fusion dehazing
Based on Ancuti & Ancuti (IEEE TIP 2013), with simplified weight maps. No haze model is estimated: two enhanced versions of the input (white-balanced, contrast-boosted) are blended with per-pixel weights in a Laplacian pyramid. Training-free.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE     = "/content/drive/MyDrive"
SRC       = f"{DRIVE}/O-HAZE"          # original O-HAZE: hazy/ + GT/
DATA_ROOT = f"{DRIVE}/O-HAZE-1600"     # resized copy, shared by all methods and the evaluation
OUT       = f"{DRIVE}/outputs/fusion_dehaze" # dehazed images, same file names as hazy/

MAX_IMAGES = 5             # first N pairs (None = all 45)
RESIZE = 1600              # long side; O-HAZE is ~3000x4000

In [ ]:
#resize hazy + GT once (skipped if DATA_ROOT already exists), same as the other method notebooks
import os, glob
from PIL import Image

def resize_long(img, long_side):
    if long_side <= 0 or max(img.size) <= long_side:
        return img
    s = long_side / max(img.size)
    return img.resize((round(img.size[0] * s), round(img.size[1] * s)), Image.BICUBIC)

if not os.path.isdir(f"{DATA_ROOT}/hazy"):
    os.makedirs(f"{DATA_ROOT}/hazy"); os.makedirs(f"{DATA_ROOT}/GT")
    pairs = list(zip(sorted(glob.glob(f"{SRC}/hazy/*")), sorted(glob.glob(f"{SRC}/GT/*"))))[:MAX_IMAGES]
    for h, g in pairs:
        hazy = resize_long(Image.open(h).convert('RGB'), RESIZE)
        gt   = Image.open(g).convert('RGB').resize(hazy.size, Image.BICUBIC)
        stem = lambda p: os.path.splitext(os.path.basename(p))[0]
        hazy.save(f"{DATA_ROOT}/hazy/{stem(h)}.png"); gt.save(f"{DATA_ROOT}/GT/{stem(g)}.png")

hazy_paths = sorted(glob.glob(f"{DATA_ROOT}/hazy/*"))
os.makedirs(OUT, exist_ok=True)
print(len(hazy_paths), "images:", [os.path.basename(p) for p in hazy_paths])

## Parameters

In [ ]:
GAMMA = 2.5                # contrast boost of the second input: I2 = GAMMA * (I - mean luminance) + mean luminance
LEVELS = 5                 # pyramid levels
EXP_SIGMA = 0.25           # exposedness weight: prefers pixels close to mid-grey (0.5)

NAME = "fusion_dehaze"

## Method

In [ ]:
def gaussian_pyr(img, levels=LEVELS):
    pyr = [img]
    for _ in range(levels - 1):
        pyr.append(cv2.pyrDown(pyr[-1]))
    return pyr


def laplacian_pyr(img, levels=LEVELS):
    g = gaussian_pyr(img, levels)
    return [g[i] - cv2.pyrUp(g[i + 1], dstsize=g[i].shape[1::-1]) for i in range(levels - 1)] + [g[-1]]


def weights(img):
    #per-pixel weight = laplacian contrast + saliency + exposedness (simplified from the paper)
    L = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)[..., 0] / 100.0
    contrast = np.abs(cv2.Laplacian(L, cv2.CV_32F))
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    saliency = np.linalg.norm(cv2.GaussianBlur(lab, (5, 5), 0) - lab.reshape(-1, 3).mean(0), axis=2)
    saliency /= saliency.max() + 1e-8
    exposed = np.exp(-((L - 0.5) ** 2) / (2 * EXP_SIGMA ** 2))
    return contrast + saliency + exposed


def dehaze(bgr):
    I = bgr.astype(np.float32) / 255

    #input 1: grey-world white balance (removes the colour cast of the haze)
    I1 = np.clip(I * (I.mean() / (I.reshape(-1, 3).mean(0) + 1e-8)), 0, 1)
    #input 2: contrast boost around the mean luminance (brings out detail in hazy regions)
    mean_lum = cv2.cvtColor(I, cv2.COLOR_BGR2GRAY).mean()
    I2 = np.clip(GAMMA * (I - mean_lum) + mean_lum, 0, 1)

    W1, W2 = weights(I1), weights(I2)
    W1, W2 = W1 / (W1 + W2 + 1e-8), W2 / (W1 + W2 + 1e-8)

    #blend: laplacian pyramids of the inputs, gaussian pyramids of the weights (avoids halos)
    fused = [w1[..., None] * l1 + w2[..., None] * l2
             for l1, l2, w1, w2 in zip(laplacian_pyr(I1), laplacian_pyr(I2), gaussian_pyr(W1), gaussian_pyr(W2))]
    out = fused[-1]
    for lev in reversed(fused[:-1]):
        out = cv2.pyrUp(out, dstsize=lev.shape[1::-1]) + lev
    return (np.clip(out, 0, 1) * 255 + 0.5).astype(np.uint8)

## Preview a single image

In [ ]:
hazy = cv2.imread(hazy_paths[0])
out = dehaze(hazy)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, im, title in zip(ax, [hazy, out], ['hazy', NAME]):
    a.imshow(im[..., ::-1]); a.set_title(title); a.axis('off')
plt.tight_layout(); plt.show()

## Run over the dataset

In [ ]:
import time
for p in hazy_paths:
    hazy = cv2.imread(p)
    t0 = time.time()
    out = dehaze(hazy)
    dt = time.time() - t0
    cv2.imwrite(f"{OUT}/{os.path.basename(p)}", out)
    print(f"{os.path.basename(p)} | {hazy.shape[1]}x{hazy.shape[0]} | {dt:.2f}s")

## Evaluate
Score the outputs with `dehazing/evaluation` (see `../DEHAZING_EVALUATION.ipynb`):
```python
per_image, summary = evaluate(DATA_ROOT, {"fusion_dehaze": OUT})
```